In [12]:
# Define the query
query = """
SELECT *
FROM [dbo].[successful_queries]
where file_id > ISNULL((SELECT max(FILE_ID)
        FROM [dbo].[tbl_errordetails_datamodel]), 0);
"""

# Execute the query
cursor.execute(query)

# Fetch all results
rows = cursor.fetchall()

# Convert result to DataFrame
columns = [column[0] for column in cursor.description]

# Convert pyodbc.Row objects to list of tuples
rows = [tuple(row) for row in rows]

In [13]:
def sp_errordetails_datamodel():
    df_error_tbl = spark.sql(""" SELECT
        file_id as file_id,
        column_name as value,
        business_rule_definition as business_rule_definition, 
        rule_no as rule_no,
        dimension as dimension, 
        severity as severity,
        attribute_group as attribute_group,  
        business_attribute_name as business_attribute_name,
        column_name as column_name,
        file_id as error_id,
        CONCAT(value, '$', error_id, '$' ,dimension, '$', attribute_group, '$',  business_attribute_name) as unique_identifier
        FROM vw_tbl_errordetails
    """)
    
    df_error_tbl.toPandas().to_sql('tbl_errordetails_datamodel', con=engine, if_exists='append', index=False)

In [14]:
# Convert result to Spark DataFrame
if rows:
    df = spark.createDataFrame(rows, schema=columns)
    df.createOrReplaceTempView('vw_tbl_errordetails')
    sp_errordetails_datamodel()
else:
    print('No New Recrds')